# Izrada aplikacija za generiranje slika (OpenAI)

LLM modeli nisu samo za generiranje teksta. Također možete generirati slike iz tekstualnih opisa. Slike kao modalitet korisne su u području MedTech, arhitekture, turizma, razvoja igara, marketinga i još mnogo toga. U ovoj lekciji radimo s današnjim **GPT Image** modelima na OpenAI platformi.

## Ciljevi učenja

Na kraju ove lekcije moći ćete:

- Objasniti što je generiranje slike i gdje je korisno.
- Razumjeti obitelj modela `gpt-image` i kako se razlikuje od zastarjelih DALL·E modela.
- Izraditi aplikaciju za generiranje slika i uređivati slike.

## Što je generiranje slika?

Modeli za generiranje slika stvaraju slike na temelju tekstualnog upita. Moderni modeli poput `gpt-image` uče odnos između teksta i slika tijekom treniranja, te zatim iterativno pretvaraju slučajnu šum u sliku koja odgovara vašem upitu.

Dvije poznate obitelji modela za slike su:

- **`gpt-image` (OpenAI)** — trenutna generacija korištena u ovoj lekciji. Podržava generiranje slika iz teksta i uređivanje slika (popunjavanje s maskom).
- **Midjourney** — popularni model treće strane s vlastitom uslugom i Discord temeljenim radnim postupkom.

> Stariji OpenAI modeli za slike — **DALL·E 2** i **DALL·E 3** — su zastarjeli. DALL·E 3 više nije dostupan za nove implementacije, a značajke poput `create_variation` postojale su samo u DALL·E 2. Za nove aplikacije koristite modele `gpt-image`.

> **Važno:** modeli `gpt-image` vraćaju generiranu sliku u obliku **base64** (`b64_json`), a ne kao URL. Vaš kod dekodira base64 niz u bajtove i sprema sliku — nema URL za preuzimanje slike.


## Izrada vaše prve aplikacije za generiranje slika

Pa što je potrebno za izradu aplikacije za generiranje slika? Trebate sljedeće biblioteke:

- **python-dotenv**, snažno se preporučuje korištenje ove biblioteke kako biste svoje tajne držali u *.env* datoteci odvojeno od koda.
- **openai**, ova biblioteka će vam služiti za interakciju s OpenAI API-jem.
- **pillow**, za rad sa slikama u Pythonu.


1. Kreirajte datoteku *.env* s sljedećim sadržajem:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Skupite gore navedene knjižnice u datoteku nazvanu *requirements.txt* ovako:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Zatim, stvorite virtualno okruženje i instalirajte knjižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows koristite sljedeće naredbe za kreiranje i aktiviranje vašeg virtualnog okruženja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte sljedeći kod u datoteku nazvanu *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Objasnimo ovaj kod:

- Najprije uvozimo potrebne biblioteke, uključujući OpenAI biblioteku, dotenv biblioteku, modul base64 i Pillow biblioteku.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Nakon toga stvaramo klijenta koji čita API ključ iz vaše ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Zatim generiramo sliku:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeli vraćaju sliku kao **base64** string u `data[0].b64_json`. Dekodiramo ga u bajtove i zapisujemo u datoteku — nije dostupna URL adresa za preuzimanje.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na kraju, otvaramo sliku i koristimo standardni preglednik slika da je prikažemo:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Više detalja o generiranju slike

Pogledajmo parametre `images.generate`:

- **model**, je model slike koji se koristi (na primjer `gpt-image-2`).
- **prompt**, je tekstualni upit koji se koristi za generiranje slike. Ovdje je "Zeko na konju, držeći lizalicu, na maglovitoj livadi gdje raste narcise".
- **size**, je veličina generirane slike (na primjer `1024x1024`, `1536x1024`, `1024x1536`, ili `"auto"`).
- **n**, je broj generiranih slika. Ovdje generiramo jednu.

> Modeli slika ne koriste `temperature` parametar — to je kontrola za generiranje teksta. Za raznolikost, pozovite API opet; za manje varijacije, učinite vaš prompt preciznijim.

## Dodatne mogućnosti generiranja slika

Vidjeli ste kako generirati sliku s nekoliko redaka Pythona. `gpt-image` modeli također mogu **uređivati** postojeću sliku. Dajući sliku, opcionalnu **masku** (koja označava područje za promjenu) i prompt, možete izmijeniti dio slike — na primjer, dodati šešir našem zeki.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika sadrži samo zeca; na završnoj slici zec ima šešir.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
